# Verifying Amplitudes and Correlation Energies

TODO:
- Create a pandas dataframe with the following structure:

structure | basis set | PySCF_Corr | Psi4_Corr | PySCF_norm_t1 | Psi4_norm_t1 | PySCF_norm_t2 | Psi4_norm_t2

- Populate the data frame with the provided code
- Use a try-except to handle linalg error, save a list of faulty structures
- Calculate the mean error for each structure basis combo


In [1]:
from shutil import copy
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from glob import glob
import psi4
from helper_CC_ML_spacial import *
import pandas as pd
import numpy as np
import glob
import os
from numpy.linalg import LinAlgError
import psi4
import pyscf
from pyscf import gto, scf, cc
from pyscf.data.elements import charge as get_atomic_number

  Threads set to 12 by Python driver.


In [2]:
def guess_charge_spin_from_xyz(xyz_text):
    atoms = [line.split()[0] for line in xyz_text.splitlines()[2:] if line.strip()]
    total_electrons = sum(get_atomic_number(atom) for atom in atoms)
    charge = 0
    spin = total_electrons % 2
    return charge, spin

In [3]:
# Path to XYZ files
xyz_folder = 'diatomics'
xyz_files = glob.glob(os.path.join(xyz_folder, '*.xyz'))

# Extract structure names and contents
structures = []
structure_contents = {}

for file_path in xyz_files:
    structure_name = os.path.splitext(os.path.basename(file_path))[0]
    structures.append(structure_name)
    with open(file_path, 'r') as f:
        structure_contents[structure_name] = f.read()

# Basis sets to iterate over

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

results = []
failures = []

for structure in structures:
    xyz_path = os.path.join(xyz_folder, f'{structure}.xyz')
    with open(xyz_path, 'r') as f:
        xyz_text = f.read()

    for basis in basis_sets:
        # Initialize default values
        psi4_corr = np.nan
        pyscf_corr = np.nan
        norm_t1_psi4 = np.nan
        norm_t2_psi4 = np.nan
        norm_t1_pyscf = np.nan
        norm_t2_pyscf = np.nan

        ##### PSI4 Processing #####
        try:
            qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
            mol_psi4 = psi4.geometry(qmol.create_psi4_string_from_molecule() + 'symmetry c1')

            psi4.core.clean()
            psi4.core.be_quiet()

            psi4.set_options({
                'basis': basis,
                'scf_type': 'pk',
                'reference': 'rohf',
                'mp2_type': 'conv',
                'e_convergence': 1e-8,
                'd_convergence': 1e-8
            })

            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            A = HelperCCEnergy(mol_psi4, rhf_e, scf_wfn, freeze_core=False)

            try:
                A.compute_energy()
                psi4_corr = A.FinalEnergy
            except LinAlgError:
                failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Corr'})

            # Extract amplitudes
            norm_t1_psi4 = np.linalg.norm(A.t1)
            norm_t2_psi4 = np.linalg.norm(A.t2)

        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'Psi4_Setup', 'error': str(e)})

        ##### PYSCF Processing #####
        try:
            charge, spin = guess_charge_spin_from_xyz(xyz_text)
            mol_pyscf = gto.Mole()
            mol_pyscf.build(atom=xyz_path, basis=basis, symmetry='c1', charge=charge, spin=spin)

            # Define active space (assuming no frozen orbitals for simplicity)
            n_frozen = 0
            active_space = range(n_frozen, mol_pyscf.nao_nr())
            scf_calc = scf.RHF(mol_pyscf) if spin == 0 else scf.UHF(mol_pyscf)
            scf_calc.kernel()

            # Compute CCSD
            ccsd_calc = cc.CCSD(scf_calc, frozen=[i for i in range(mol_pyscf.nao_nr()) if i not in active_space]).run()
            pyscf_corr = ccsd_calc.e_corr

            # t1 norm
            t1 = ccsd_calc.t1
            if isinstance(t1, tuple):
                norm_t1_pyscf = np.sqrt(sum(np.linalg.norm(t)**2 for t in t1))
            else:
                norm_t1_pyscf = np.linalg.norm(t1)

            # t2 norm
            t2 = ccsd_calc.t2
            if isinstance(t2, tuple):
                norm_t2_pyscf = np.sqrt(sum(np.linalg.norm(t)**2 for t in t2))
            else:
                norm_t2_pyscf = np.linalg.norm(t2)

        except LinAlgError:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Corr'})
        except Exception as e:
            failures.append({'structure': structure, 'basis set': basis, 'method': 'PySCF_Setup', 'error': str(e)})

        # Store results
        results.append({
            'structure': structure,
            'basis set': basis,
            'PySCF_Corr': pyscf_corr,
            'Psi4_Corr': psi4_corr,
            'PySCF_norm_t1': norm_t1_pyscf,
            'Psi4_norm_t1': norm_t1_psi4,
            'PySCF_norm_t2': norm_t2_pyscf,
            'Psi4_norm_t2': norm_t2_psi4
        })

# Create and display final DataFrame
df = pd.DataFrame(results)
print("\nFinal DataFrame:")
print(df)

if failures:
    print("\nFailures Encountered:")
    for failure in failures:
        print(f"Structure: {failure['structure']}, Basis Set: {failure['basis set']}, Method: {failure['method']}, Error: {failure.get('error', '')}")

# Optional: Save results to CSV
# df.to_csv('amplitude_correlation_results.csv', index=False)


Computing RHF reference.


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 10 basis functions.
(10, 10)
(10, 10)
Building initial guess...

..initialized CCSD in 0.046 seconds.

CCSD Iteration   0: CCSD correlation = -0.195012162757378   dE =  1.95012E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.230232040574517   dE = -3.52199E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.278467003636465   dE = -4.82350E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.277269813342813   dE =  1.19719E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.305898084377869   dE = -2.86283E-02   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.324524743464933   dE = -1.86267E-02   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.312245838662287   dE =  1.22789E-02   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.315864215065470   dE = -3.61838E-03   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.315997540226707   dE = -1.33325E-04   DIIS = 7


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 28 basis functions.
(28, 28)
(28, 28)
Building initial guess...

..initialized CCSD in 0.534 seconds.

CCSD Iteration   0: CCSD correlation = -0.272592287405190   dE =  2.72592E-01   MP2


...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 6 basis functions.
(6, 6)
(6, 6)
Building initial guess...

..initialized CCSD in 0.009 seconds.

CCSD Iteration   0: CCSD correlation = -0.012928074241387   dE =  1.29281E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.022656495290520   dE = -9.72842E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.025988079937004   dE = -3.33158E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.027990904654932   dE = -2.00282E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.028849668232054   dE = -8.58764E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.029026485378463   dE = -1.76817E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.029000170530786   dE =  2.63148E-05   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.029004633286972   dE = -4.46276E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.029004440093496   dE =  1.93193E-07   DIIS = 7
CCSD 

/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 19 basis functions.
(19, 19)
(19, 19)
Building initial guess...

..initialized CCSD in 0.163 seconds.

CCSD Iteration   0: CCSD correlation = -0.024926679373109   dE =  2.49267E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.039133931770580   dE = -1.42073E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.043584969489971   dE = -4.45104E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.046256790089471   dE = -2.67182E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.047411069932371   dE = -1.15428E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.047590004406836   dE = -1.78934E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.047587804282730   dE =  2.20012E-06   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.047586638126600   dE =  1.16616E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.047587857773846   dE = -1.21965E-06   DIIS = 7


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 32 basis functions.
(32, 32)
(32, 32)
Building initial guess...

..initialized CCSD in 0.369 seconds.

CCSD Iteration   0: CCSD correlation = -0.025505900335135   dE =  2.55059E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.039807440432760   dE = -1.43015E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.044528971673814   dE = -4.72153E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.047615359582245   dE = -3.08639E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.048777063721947   dE = -1.16170E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.048954595505387   dE = -1.77532E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.048953511810203   dE =  1.08370E-06   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.048952500771369   dE =  1.01104E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.048953563050711   dE = -1.06228E-06   DIIS = 7


In [4]:
failures_df =pd.DataFrame(failures)

In [5]:
failures_df.to_csv('error_log.csv', index=False)

In [6]:
df.to_csv('results.csv', index=False)

In [7]:
# Filter to valid data
valid_corr = df.dropna(subset=['PySCF_Corr', 'Psi4_Corr'])

# Calculate MAE
mae = np.mean(np.abs(valid_corr['PySCF_Corr'] - valid_corr['Psi4_Corr']))
print(f"Mean Absolute Error: {mae:.6f} Hartree")


Mean Absolute Error: 0.025569 Hartree


## Parameters and Frozen Orbitals

| **Psi4** Option  | **Meaning**                              | **PySCF Equivalent**                                                                    |
| ---------------- | ---------------------------------------- | --------------------------------------------------------------------------------------- |
| `scf_type: pk`   | Full integral storage (slow but precise) | PySCF *always* uses full integral evaluation (AO or MO), so no direct need to set this. |
| `reference: rhf` | Restricted HF (singlet)                  | Use `scf.RHF(mol)` in PySCF or modify it as needed                                      |
| `mp2_type: conv` | Conventional MP2 (not DF-MP2)            | PySCF uses conventional CCSD by default. For MP2, use `mp.MP2(mf, auxbasis=None)`.      |
| `e_convergence`  | Energy convergence (1e-8)                | Set via `conv_tol` in PySCF: `mf.conv_tol = 1e-8`                                       |
| `d_convergence`  | Density matrix convergence (1e-8)        | Set via `conv_tol_grad` in PySCF: `mf.conv_tol_grad = 1e-8`                             |


In [ ]:
# Convergence and reference state
scf_calc = scf.RHF(mol_pyscf)
scf_calc.conv_tol = 1e-8           # Match Psi4 e_convergence
scf_calc.conv_tol_grad = 1e-8      # Match Psi4 d_convergence
scf_calc.kernel()


# ROHF 
mf = scf.ROHF(mol_pyscf)
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-8
mf.kernel()


In [ ]:
# To Freeze Orbitals, you have to do it manually:
ccsd = cc.CCSD(mf, frozen=[0, 1])  # Freeze orbitals 0 and 1

